<a href="https://colab.research.google.com/github/Krishna101010101010/Samudrika-Underwater-Image-Enhancement-for-Maritime-Security/blob/main/Samudrika%20Funie-GAN%20Training%20Pipeline%20%26%20Dataset%20Merger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
  """
  Samudrika-v1 — FUnIE-GAN Dataset Merger
  ========================================
  Merges UIEB + Large-Scale-Underwater-datasets (LSUI) into one
  training-ready folder: FUnIE-Gan_merged_dataset/

  Expected input structure (inside your Kaggle/Colab dataset path):
    datasets/
      UIEB/
        raw-890/          ← distorted images
        reference-890/    ← ground truth images
        challenging-60/   ← NO reference → skipped intentionally
      Large-Scale-Underwater-datasets/
        input/            ← distorted images
        GT/               ← ground truth images

  Output structure:
    FUnIE-Gan_merged_dataset/
      distorted/          ← all raw/input images, prefixed to avoid collision
      ground_truth/       ← all reference/GT images, same prefix + filename
      split/
        train.txt         ← 80% of filenames
        val.txt           ← 10% of filenames
        test.txt          ← 10% of filenames
      merge_report.txt    ← summary of what was merged
  """

  import os
  import shutil
  import random
  from pathlib import Path
  from datetime import datetime

  # ─────────────────────────────────────────────
  # CONFIG — adjust BASE_DIR to wherever your
  # datasets/ folder lives (Kaggle input path)
  # ─────────────────────────────────────────────
  BASE_DIR    = Path("/content/drive/MyDrive/Funie-Gan Dataset")
  OUTPUT_DIR  = Path("/content/FUnIE-Gan_merged_dataset")

  TRAIN_RATIO = 0.80
  VAL_RATIO   = 0.10
  TEST_RATIO  = 0.10
  RANDOM_SEED = 42

  VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

  # ─────────────────────────────────────────────
  # Dataset definitions
  # Each entry: (prefix, distorted_dir, gt_dir)
  # prefix prevents filename collisions on merge
  # ─────────────────────────────────────────────
DATASETS = [
    (
        "uieb",
        BASE_DIR / "UIEB - Underwater Image Enhancement Benchmark Dataset" / "raw-890",
        BASE_DIR / "UIEB - Underwater Image Enhancement Benchmark Dataset" / "reference-890",
    ),
    (
        "lsui",
        BASE_DIR / "LSUI - Large Scale Underwater Image Dataset" / "input",
        BASE_DIR / "LSUI - Large Scale Underwater Image Dataset" / "GT",
    ),
]


  # ─────────────────────────────────────────────
  # HELPERS
  # ─────────────────────────────────────────────

  def is_image(path: Path) -> bool:
      return path.suffix.lower() in VALID_EXTS


  def find_gt_match(distorted_path: Path, gt_dir: Path) -> Path | None:
      """
      Try to find the matching ground truth for a distorted image.
      Handles cases where extensions might differ (e.g. .jpg vs .png).
      Returns the GT path if found, else None.
      """
      stem = distorted_path.stem

      # Exact match first (same extension)
      exact = gt_dir / distorted_path.name
      if exact.exists():
          return exact

      # Try all valid extensions if exact not found
      for ext in VALID_EXTS:
          candidate = gt_dir / (stem + ext)
          if candidate.exists():
              return candidate

      return None


  def scan_dataset(prefix: str, distorted_dir: Path, gt_dir: Path):
      """
      Scans a dataset folder and returns list of (new_name, dist_path, gt_path).
      Skips any distorted image that has no GT match.
      """
      if not distorted_dir.exists():
          print(f"  [WARN] distorted dir not found: {distorted_dir}")
          return [], []

      if not gt_dir.exists():
          print(f"  [WARN] GT dir not found: {gt_dir}")
          return [], []

      pairs   = []
      skipped = []

      for dist_path in sorted(distorted_dir.iterdir()):
          if not is_image(dist_path):
              continue

          gt_path = find_gt_match(dist_path, gt_dir)
          if gt_path is None:
              skipped.append(dist_path.name)
              continue

          # Build new unified name: prefix_originalname.ext
          # Use the distorted image's extension for consistency
          new_name = f"{prefix}_{dist_path.name}"
          pairs.append((new_name, dist_path, gt_path))

      return pairs, skipped


  def copy_pairs(pairs, dist_out: Path, gt_out: Path, dataset_label: str):
      """Copy paired images to output dirs with progress reporting."""
      copied = 0
      for i, (new_name, dist_path, gt_path) in enumerate(pairs):
          dst_dist = dist_out / new_name
          dst_gt   = gt_out   / new_name

          shutil.copy2(dist_path, dst_dist)
          shutil.copy2(gt_path,   dst_gt)
          copied += 1

          if (i + 1) % 200 == 0:
              print(f"    [{dataset_label}] copied {i+1}/{len(pairs)} pairs...")

      return copied


  def write_splits(all_names, split_dir: Path):
      """Create train/val/test split .txt files."""
      random.seed(RANDOM_SEED)
      shuffled = all_names[:]
      random.shuffle(shuffled)

      n      = len(shuffled)
      n_train = int(n * TRAIN_RATIO)
      n_val   = int(n * VAL_RATIO)

      train_names = shuffled[:n_train]
      val_names   = shuffled[n_train : n_train + n_val]
      test_names  = shuffled[n_train + n_val :]

      split_dir.mkdir(parents=True, exist_ok=True)
      (split_dir / "train.txt").write_text("\n".join(train_names))
      (split_dir / "val.txt"  ).write_text("\n".join(val_names))
      (split_dir / "test.txt" ).write_text("\n".join(test_names))

      return len(train_names), len(val_names), len(test_names)


  # ─────────────────────────────────────────────
  # MAIN
  # ─────────────────────────────────────────────

  def main():
      print("=" * 60)
      print("  Samudrika-v1 — FUnIE-GAN Dataset Merger")
      print("=" * 60)

      # Create output dirs
      dist_out  = OUTPUT_DIR / "distorted"
      gt_out    = OUTPUT_DIR / "ground_truth"
      split_dir = OUTPUT_DIR / "split"

      for d in [dist_out, gt_out, split_dir]:
          d.mkdir(parents=True, exist_ok=True)

      all_pairs   = []
      report_lines = [
          f"Samudrika-v1 FUnIE-GAN Merge Report",
          f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
          f"Output dir: {OUTPUT_DIR}",
          "=" * 50,
          "",
      ]

      # ── Scan each dataset ──
      for prefix, dist_dir, gt_dir in DATASETS:
          print(f"\n[{prefix.upper()}]")
          print(f"  distorted : {dist_dir}")
          print(f"  ground_truth: {gt_dir}")

          pairs, skipped = scan_dataset(prefix, dist_dir, gt_dir)

          print(f"  Found   : {len(pairs)} valid pairs")
          if skipped:
              print(f"  Skipped : {len(skipped)} unpaired files → {skipped[:5]}{'...' if len(skipped)>5 else ''}")

          report_lines += [
              f"[{prefix.upper()}]",
              f"  Source distorted : {dist_dir}",
              f"  Source GT        : {gt_dir}",
              f"  Valid pairs      : {len(pairs)}",
              f"  Skipped (no GT)  : {len(skipped)}",
              "",
          ]

          all_pairs.extend(pairs)

      # ── Copy all pairs ──
      print(f"\n[COPYING] {len(all_pairs)} total pairs to output...")
      for prefix, _, _ in DATASETS:
          dataset_pairs = [(n, d, g) for n, d, g in all_pairs if n.startswith(prefix)]
          copied = copy_pairs(dataset_pairs, dist_out, gt_out, prefix.upper())
          print(f"  [{prefix.upper()}] {copied} pairs copied ✓")

      # ── Verify output counts match ──
      dist_count = len([f for f in dist_out.iterdir() if is_image(f)])
      gt_count   = len([f for f in gt_out.iterdir()   if is_image(f)])

      print(f"\n[VERIFY]")
      print(f"  distorted/    : {dist_count} images")
      print(f"  ground_truth/ : {gt_count} images")

      if dist_count != gt_count:
          print(f"  [ERROR] Count mismatch! dist={dist_count} gt={gt_count}")
          report_lines.append(f"ERROR: count mismatch dist={dist_count} gt={gt_count}")
      else:
          print(f"  All pairs balanced ✓")

      # ── Spot-check: verify a few GT filenames match distorted ──
      dist_names = sorted([f.name for f in dist_out.iterdir() if is_image(f)])
      gt_names   = sorted([f.name for f in gt_out.iterdir()   if is_image(f)])

      mismatches = [n for n in dist_names if n not in set(gt_names)]
      if mismatches:
          print(f"  [WARN] {len(mismatches)} filenames in distorted/ have no match in ground_truth/")
          print(f"    First 5: {mismatches[:5]}")
      else:
          print(f"  Filename pairing verified ✓")

      # ── Write splits ──
      all_names = [new_name for new_name, _, _ in all_pairs]
      n_train, n_val, n_test = write_splits(all_names, split_dir)

      print(f"\n[SPLITS] seed={RANDOM_SEED}")
      print(f"  train : {n_train} ({n_train/len(all_names)*100:.1f}%)")
      print(f"  val   : {n_val}   ({n_val/len(all_names)*100:.1f}%)")
      print(f"  test  : {n_test}  ({n_test/len(all_names)*100:.1f}%)")

      # ── Write report ──
      report_lines += [
          "MERGE SUMMARY",
          f"  Total pairs merged : {len(all_pairs)}",
          f"  distorted count    : {dist_count}",
          f"  ground_truth count : {gt_count}",
          f"  Filename mismatches: {len(mismatches)}",
          "",
          "SPLITS",
          f"  train : {n_train}",
          f"  val   : {n_val}",
          f"  test  : {n_test}",
      ]

      report_path = OUTPUT_DIR / "merge_report.txt"
      report_path.write_text("\n".join(report_lines))

      print(f"\n[DONE]")
      print(f"  Output : {OUTPUT_DIR}")
      print(f"  Report : {report_path}")
      print("=" * 60)
      print(f"\n  Final structure:")
      print(f"  FUnIE-Gan_merged_dataset/")
      print(f"    distorted/       ({dist_count} images)")
      print(f"    ground_truth/    ({gt_count} images)")
      print(f"    split/")
      print(f"      train.txt      ({n_train} filenames)")
      print(f"      val.txt        ({n_val} filenames)")
      print(f"      test.txt       ({n_test} filenames)")
      print(f"    merge_report.txt")


  if __name__ == "__main__":
      main()

  Samudrika-v1 — FUnIE-GAN Dataset Merger

[UIEB]
  distorted : /content/drive/MyDrive/Funie-Gan Dataset/UIEB - Underwater Image Enhancement Benchmark Dataset/raw-890
  ground_truth: /content/drive/MyDrive/Funie-Gan Dataset/UIEB - Underwater Image Enhancement Benchmark Dataset/reference-890
  Found   : 890 valid pairs

[LSUI]
  distorted : /content/drive/MyDrive/Funie-Gan Dataset/LSUI - Large Scale Underwater Image Dataset/input
  ground_truth: /content/drive/MyDrive/Funie-Gan Dataset/LSUI - Large Scale Underwater Image Dataset/GT
  Found   : 4279 valid pairs

[COPYING] 5169 total pairs to output...
    [UIEB] copied 200/890 pairs...
    [UIEB] copied 400/890 pairs...
    [UIEB] copied 600/890 pairs...
    [UIEB] copied 800/890 pairs...
  [UIEB] 890 pairs copied ✓
    [LSUI] copied 200/4279 pairs...
    [LSUI] copied 400/4279 pairs...
    [LSUI] copied 600/4279 pairs...
    [LSUI] copied 800/4279 pairs...
    [LSUI] copied 1000/4279 pairs...
    [LSUI] copied 1200/4279 pairs...
    [LS

In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/FUnIE-Gan_merged_dataset")


In [ ]:
# If merged dataset is still in /content/ (temporary):
DATASET_DIR = Path('/content/FUnIE-Gan_merged_dataset')

# If you copied it to Drive:
DATASET_DIR = Path('/content/drive/MyDrive/FUnIE-Gan_merged_dataset')

# Samudrika-v1 — FUnIE-GAN Training
### Pipeline: Raw underwater image → FUnIE-GAN enhancement → YOLOv11 detection

### Dataset: 5169 paired images (UIEB 890 + LSUI 4279)

### Target: 50 epochs · batch 8 · T4 GPU · ~2.5 hours

Checklist before running
 Runtime → Change runtime type → T4 GPU
 Mount Google Drive (Cell 1)
 Verify dataset path exists (Cell 2)
 Run all cells top to bottom


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Mount Drive + Install dependencies
# ═══════════════════════════════════════════════════════════

!pip install -q torchmetrics Pillow tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 30.0 MB/s eta 0:00:00


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CONFIG  ← only cell you ever need to edit
# ═══════════════════════════════════════════════════════════
from pathlib import Path

# ── Dataset (update if you saved merged dataset to Drive) ──
# If merged dataset is in /content/ (temporary), set:
DATASET_DIR = Path('/content/FUnIE-Gan_merged_dataset')
# If you copied it to Drive, change to:
# DATASET_DIR = Path('/content/drive/MyDrive/FUnIE-Gan_merged_dataset')

# ── Output — always save to Drive so checkpoints survive session end ──
OUTPUT_DIR  = Path('/content/drive/MyDrive/Samudrika-v1/funiegan_training')

# ── Training hyperparameters ──
EPOCHS      = 50
BATCH_SIZE  = 8
IMG_SIZE    = 256          # FUnIE-GAN standard
LR          = 0.0002       # Adam learning rate
BETA1       = 0.5          # Adam beta1 (standard for GANs)
LAMBDA_L1   = 100.0        # L1 loss weight
LAMBDA_PERC = 10.0         # Perceptual loss weight
DECAY_EPOCH = 25           # Start LR linear decay at this epoch
SAVE_EVERY  = 5            # Save checkpoint every N epochs
RANDOM_SEED = 42

# ── Device ──
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device      : {DEVICE}')
print(f'Dataset dir : {DATASET_DIR}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Epochs      : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  Image: {IMG_SIZE}x{IMG_SIZE}')

# Create output dirs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'checkpoints').mkdir(exist_ok=True)
(OUTPUT_DIR / 'samples').mkdir(exist_ok=True)

Device      : cuda
Dataset dir : /content/FUnIE-Gan_merged_dataset
Output dir  : /content/drive/MyDrive/Samudrika-v1/funiegan_training
Epochs      : 50  |  Batch: 8  |  Image: 256x256


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Verify dataset
# ═══════════════════════════════════════════════════════════
import os

dist_dir = DATASET_DIR / 'distorted'
gt_dir   = DATASET_DIR / 'ground_truth'
split_dir = DATASET_DIR / 'split'

assert dist_dir.exists(),  f'distorted/ not found at {dist_dir}'
assert gt_dir.exists(),    f'ground_truth/ not found at {gt_dir}'
assert split_dir.exists(), f'split/ not found at {split_dir}'

dist_imgs = sorted(list(dist_dir.glob('*')))
gt_imgs   = sorted(list(gt_dir.glob('*')))

print(f'distorted/    : {len(dist_imgs)} images')
print(f'ground_truth/ : {len(gt_imgs)} images')
print(f'train.txt     : {len(open(split_dir/"train.txt").readlines())} entries')
print(f'val.txt       : {len(open(split_dir/"val.txt").readlines())} entries')
print(f'test.txt      : {len(open(split_dir/"test.txt").readlines())} entries')
assert len(dist_imgs) == len(gt_imgs), 'ERROR: image count mismatch!'
print('\nDataset verified ✓')

distorted/    : 5169 images
ground_truth/ : 5169 images
train.txt     : 4135 entries
val.txt       : 516 entries
test.txt      : 518 entries

Dataset verified ✓


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Dataset & DataLoader
# ═══════════════════════════════════════════════════════════
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class UnderwaterPairDataset(Dataset):
    def __init__(self, split_file: Path, dist_dir: Path, gt_dir: Path, img_size: int):
        self.dist_dir = dist_dir
        self.gt_dir   = gt_dir
        self.names    = [l.strip() for l in open(split_file).readlines() if l.strip()]
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # → [-1, 1]
        ])

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        # Try exact name first, then glob by stem if extension differs
        dist_path = self.dist_dir / name
        gt_path   = self.gt_dir   / name

        if not dist_path.exists():
            matches = list(self.dist_dir.glob(Path(name).stem + '.*'))
            dist_path = matches[0] if matches else dist_path
        if not gt_path.exists():
            matches = list(self.gt_dir.glob(Path(name).stem + '.*'))
            gt_path = matches[0] if matches else gt_path

        dist_img = Image.open(dist_path).convert('RGB')
        gt_img   = Image.open(gt_path).convert('RGB')

        return {
            'A': self.transform(dist_img),   # distorted
            'B': self.transform(gt_img),     # ground truth
            'name': name
        }


# Build datasets
train_dataset = UnderwaterPairDataset(split_dir/'train.txt', dist_dir, gt_dir, IMG_SIZE)
val_dataset   = UnderwaterPairDataset(split_dir/'val.txt',   dist_dir, gt_dir, IMG_SIZE)
test_dataset  = UnderwaterPairDataset(split_dir/'test.txt',  dist_dir, gt_dir, IMG_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}  ({len(train_dataset)} images)')
print(f'Val batches   : {len(val_loader)}  ({len(val_dataset)} images)')
print(f'Test images   : {len(test_dataset)}')

# Quick sanity check — load one batch
batch = next(iter(train_loader))
print(f'Batch A shape : {batch["A"].shape}  (distorted)')
print(f'Batch B shape : {batch["B"].shape}  (ground truth)')
print(f'Value range   : [{batch["A"].min():.2f}, {batch["A"].max():.2f}]  (should be ~[-1, 1])')

Train batches : 517  (4135 images)
Val batches   : 65  (516 images)
Test images   : 518
Batch A shape : torch.Size([8, 3, 256, 256])  (distorted)
Batch B shape : torch.Size([8, 3, 256, 256])  (ground truth)
Value range   : [-1.00, 1.00]  (should be ~[-1, 1])


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Model Architecture (U-Net Generator + PatchGAN Discriminator)
# ═══════════════════════════════════════════════════════════
import torch
import torch.nn as nn


class UNetBlock(nn.Module):
    """Single encoder or decoder block for U-Net."""
    def __init__(self, in_ch, out_ch, down=True, use_norm=True, dropout=False):
        super().__init__()
        layers = []
        if down:
            layers += [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        else:
            layers += [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if use_norm:
            layers += [nn.InstanceNorm2d(out_ch)]
        layers += [nn.LeakyReLU(0.2, True) if down else nn.ReLU(True)]
        if dropout:
            layers += [nn.Dropout(0.5)]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetGenerator(nn.Module):
    """FUnIE-GAN U-Net Generator with skip connections."""
    def __init__(self, in_ch=3, out_ch=3, nf=64):
        super().__init__()
        # ── Encoder ──
        self.e1 = nn.Sequential(nn.Conv2d(in_ch, nf, 4, 2, 1), nn.LeakyReLU(0.2, True))  # no norm on first
        self.e2 = UNetBlock(nf,    nf*2)
        self.e3 = UNetBlock(nf*2,  nf*4)
        self.e4 = UNetBlock(nf*4,  nf*8)
        self.e5 = UNetBlock(nf*8,  nf*8)
        self.e6 = UNetBlock(nf*8,  nf*8)
        self.e7 = UNetBlock(nf*8,  nf*8)
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(nf*8, nf*8, 4, 2, 1), nn.ReLU(True)
        )
        # ── Decoder (with skip connections → in_ch doubled) ──
        self.d1 = UNetBlock(nf*8,   nf*8,  down=False, dropout=True)
        self.d2 = UNetBlock(nf*16,  nf*8,  down=False, dropout=True)
        self.d3 = UNetBlock(nf*16,  nf*8,  down=False, dropout=True)
        self.d4 = UNetBlock(nf*16,  nf*8,  down=False)
        self.d5 = UNetBlock(nf*16,  nf*4,  down=False)
        self.d6 = UNetBlock(nf*8,   nf*2,  down=False)
        self.d7 = UNetBlock(nf*4,   nf,    down=False)
        self.out = nn.Sequential(
            nn.ConvTranspose2d(nf*2, out_ch, 4, 2, 1),
            nn.Tanh()  # output in [-1, 1]
        )

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        e5 = self.e5(e4)
        e6 = self.e6(e5)
        e7 = self.e7(e6)
        b  = self.bottleneck(e7)
        d1 = self.d1(b)
        d2 = self.d2(torch.cat([d1, e7], 1))
        d3 = self.d3(torch.cat([d2, e6], 1))
        d4 = self.d4(torch.cat([d3, e5], 1))
        d5 = self.d5(torch.cat([d4, e4], 1))
        d6 = self.d6(torch.cat([d5, e3], 1))
        d7 = self.d7(torch.cat([d6, e2], 1))
        return self.out(torch.cat([d7, e1], 1))


class PatchGANDiscriminator(nn.Module):
    """70x70 PatchGAN Discriminator."""
    def __init__(self, in_ch=6, nf=64):  # in_ch=6: concat real_A + real_B/fake_B
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch, nf,    4, 2, 1),        nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf,    nf*2,  4, 2, 1, bias=False), nn.InstanceNorm2d(nf*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*2,  nf*4,  4, 2, 1, bias=False), nn.InstanceNorm2d(nf*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*4,  nf*8,  4, 1, 1, bias=False), nn.InstanceNorm2d(nf*8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*8,  1,     4, 1, 1)  # output: patch prediction map
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], 1))


def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


# Instantiate
generator     = UNetGenerator().to(DEVICE)
discriminator = PatchGANDiscriminator().to(DEVICE)
generator.apply(init_weights)
discriminator.apply(init_weights)

# Parameter counts
g_params = sum(p.numel() for p in generator.parameters()) / 1e6
d_params = sum(p.numel() for p in discriminator.parameters()) / 1e6
print(f'Generator     : {g_params:.2f}M parameters')
print(f'Discriminator : {d_params:.2f}M parameters')
print('Models initialized ✓')

Generator     : 54.40M parameters
Discriminator : 2.77M parameters
Models initialized ✓


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Loss Functions
# ═══════════════════════════════════════════════════════════
import torch
import torch.nn as nn
from torchvision import models

# ── LSGAN loss (more stable than vanilla BCE) ──
criterion_gan = nn.MSELoss()

# ── L1 loss (pixel-level accuracy) ──
criterion_l1  = nn.L1Loss()

# ── Perceptual loss (VGG16 relu3_3 features) ──
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        # relu3_3 = layers up to index 15
        self.feature_extractor = nn.Sequential(*list(vgg.children())[:16]).eval()
        for p in self.feature_extractor.parameters():
            p.requires_grad = False
        self.feature_extractor = self.feature_extractor.to(DEVICE)

    def forward(self, pred, target):
        # Input is [-1,1], VGG expects [0,1] normalized with ImageNet stats
        pred_01   = (pred   + 1) / 2
        target_01 = (target + 1) / 2
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(pred.device)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(pred.device)
        pred_vgg   = (pred_01   - mean) / std
        target_vgg = (target_01 - mean) / std
        return nn.functional.l1_loss(
            self.feature_extractor(pred_vgg),
            self.feature_extractor(target_vgg)
        )

criterion_perc = PerceptualLoss()
print('Loss functions initialized ✓')
print('  GAN loss      : LSGAN (MSE)')
print('  L1 loss       : weight', LAMBDA_L1)
print('  Perceptual    : VGG16 relu3_3, weight', LAMBDA_PERC)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 101MB/s]


Loss functions initialized ✓
  GAN loss      : LSGAN (MSE)
  L1 loss       : weight 100.0
  Perceptual    : VGG16 relu3_3, weight 10.0


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Optimizers & LR Schedulers
# ═══════════════════════════════════════════════════════════
import torch.optim as optim

opt_G = optim.Adam(generator.parameters(),     lr=LR, betas=(BETA1, 0.999))
opt_D = optim.Adam(discriminator.parameters(), lr=LR, betas=(BETA1, 0.999))

# Linear decay: keep LR flat until DECAY_EPOCH, then decay to 0 by EPOCHS
def lr_lambda(epoch):
    if epoch < DECAY_EPOCH:
        return 1.0
    return max(0.0, 1.0 - (epoch - DECAY_EPOCH) / (EPOCHS - DECAY_EPOCH))

sched_G = optim.lr_scheduler.LambdaLR(opt_G, lr_lambda)
sched_D = optim.lr_scheduler.LambdaLR(opt_D, lr_lambda)

print('Optimizers    : Adam  lr={LR}  beta1={BETA1}')
print(f'LR schedule   : flat for {DECAY_EPOCH} epochs, then linear decay to 0')

Optimizers    : Adam  lr={LR}  beta1={BETA1}
LR schedule   : flat for 25 epochs, then linear decay to 0


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Checkpoint: Save & Resume
# ═══════════════════════════════════════════════════════════
import torch

CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
BEST_MODEL     = OUTPUT_DIR / 'funiegan_best.pth'

def save_checkpoint(epoch, g_loss, d_loss, psnr, ssim, is_best=False):
    ckpt = {
        'epoch'      : epoch,
        'g_state'    : generator.state_dict(),
        'd_state'    : discriminator.state_dict(),
        'opt_g'      : opt_G.state_dict(),
        'opt_d'      : opt_D.state_dict(),
        'sched_g'    : sched_G.state_dict(),
        'sched_d'    : sched_D.state_dict(),
        'g_loss'     : g_loss,
        'd_loss'     : d_loss,
        'psnr'       : psnr,
        'ssim'       : ssim,
    }
    path = CHECKPOINT_DIR / f'epoch_{epoch:03d}.pth'
    torch.save(ckpt, path)
    if is_best:
        torch.save(ckpt, BEST_MODEL)
        print(f'  ★ Best model saved → funiegan_best.pth  (PSNR={psnr:.2f} SSIM={ssim:.4f})')


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)
    generator.load_state_dict(ckpt['g_state'])
    discriminator.load_state_dict(ckpt['d_state'])
    opt_G.load_state_dict(ckpt['opt_g'])
    opt_D.load_state_dict(ckpt['opt_d'])
    sched_G.load_state_dict(ckpt['sched_g'])
    sched_D.load_state_dict(ckpt['sched_d'])
    print(f'Resumed from epoch {ckpt["epoch"]}  PSNR={ckpt["psnr"]:.2f}  SSIM={ckpt["ssim"]:.4f}')
    return ckpt['epoch']


# ── Auto-resume: find latest checkpoint if it exists ──
START_EPOCH = 0
existing = sorted(CHECKPOINT_DIR.glob('epoch_*.pth'))
if existing:
    print(f'Found {len(existing)} checkpoint(s). Resuming from latest...')
    START_EPOCH = load_checkpoint(existing[-1])
else:
    print('No checkpoints found — starting fresh from epoch 0')

No checkpoints found — starting fresh from epoch 0


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Validation (PSNR + SSIM)
# ═══════════════════════════════════════════════════════════
import torch
from torchmetrics.functional import (
    peak_signal_noise_ratio as psnr_fn,
    structural_similarity_index_measure as ssim_fn
)

@torch.no_grad()
def validate():
    generator.eval()
    total_l1, total_psnr, total_ssim = 0.0, 0.0, 0.0

    for batch in val_loader:
        real_A = batch['A'].to(DEVICE)  # distorted
        real_B = batch['B'].to(DEVICE)  # ground truth
        fake_B = generator(real_A)      # enhanced

        # L1 in [-1,1] space
        total_l1 += criterion_l1(fake_B, real_B).item()

        # PSNR/SSIM require [0,1] space
        fake_01 = (fake_B + 1) / 2
        real_01 = (real_B + 1) / 2
        total_psnr += psnr_fn(fake_01, real_01, data_range=1.0).item()
        total_ssim += ssim_fn(fake_01, real_01, data_range=1.0).item()

    generator.train()
    n = len(val_loader)
    return total_l1/n, total_psnr/n, total_ssim/n

print('Validation function ready')
print('Metrics: L1 loss · PSNR (target >22 dB) · SSIM (target >0.80)')

Validation function ready
Metrics: L1 loss · PSNR (target >22 dB) · SSIM (target >0.80)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Sample visualization helper
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import torchvision

@torch.no_grad()
def save_samples(epoch, n=4):
    generator.eval()
    batch = next(iter(val_loader))
    real_A = batch['A'][:n].to(DEVICE)
    real_B = batch['B'][:n].to(DEVICE)
    fake_B = generator(real_A)

    # Denormalize [-1,1] → [0,1]
    def denorm(t): return (t.clamp(-1,1) + 1) / 2

    grid = torchvision.utils.make_grid(
        torch.cat([denorm(real_A), denorm(fake_B), denorm(real_B)], 0),
        nrow=n, padding=2
    )
    fig, ax = plt.subplots(1, 1, figsize=(16, 5))
    ax.imshow(grid.cpu().permute(1,2,0))
    ax.set_title(f'Epoch {epoch} — Top: Distorted | Middle: Enhanced | Bottom: Ground Truth')
    ax.axis('off')
    save_path = OUTPUT_DIR / 'samples' / f'epoch_{epoch:03d}.png'
    plt.savefig(save_path, bbox_inches='tight', dpi=100)
    plt.show()
    plt.close()
    generator.train()

print('Sample visualization helper ready')

Sample visualization helper ready


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — TRAINING LOOP
# ═══════════════════════════════════════════════════════════
import time
from tqdm import tqdm

# History for plotting
history = {
    'epoch': [], 'g_loss': [], 'd_loss': [],
    'val_l1': [], 'psnr': [], 'ssim': []
}

best_psnr   = -1.0
train_start = time.time()

# Target labels for LSGAN
def real_label(size): return torch.ones(size).to(DEVICE)
def fake_label(size): return torch.zeros(size).to(DEVICE)

print(f'Starting training from epoch {START_EPOCH+1} to {EPOCHS}')
print(f'Checkpoints every {SAVE_EVERY} epochs → {CHECKPOINT_DIR}')
print(f'Best model     → {BEST_MODEL}')
print('=' * 65)

for epoch in range(START_EPOCH + 1, EPOCHS + 1):
    epoch_start = time.time()
    g_loss_sum, d_loss_sum = 0.0, 0.0

    generator.train()
    discriminator.train()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:03d}/{EPOCHS}', leave=False)

    for batch in pbar:
        real_A = batch['A'].to(DEVICE)  # distorted
        real_B = batch['B'].to(DEVICE)  # ground truth
        fake_B = generator(real_A)      # enhanced (generated)

        # ── Train Discriminator ──
        opt_D.zero_grad()
        pred_real = discriminator(real_A, real_B)
        loss_D_real = criterion_gan(pred_real, real_label(pred_real.shape))
        pred_fake = discriminator(real_A, fake_B.detach())
        loss_D_fake = criterion_gan(pred_fake, fake_label(pred_fake.shape))
        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        opt_D.step()

        # ── Train Generator ──
        opt_G.zero_grad()
        pred_fake_g = discriminator(real_A, fake_B)
        loss_G_gan  = criterion_gan(pred_fake_g, real_label(pred_fake_g.shape))
        loss_G_l1   = criterion_l1(fake_B, real_B) * LAMBDA_L1
        loss_G_perc = criterion_perc(fake_B, real_B) * LAMBDA_PERC
        loss_G = loss_G_gan + loss_G_l1 + loss_G_perc
        loss_G.backward()
        opt_G.step()

        g_loss_sum += loss_G.item()
        d_loss_sum += loss_D.item()
        pbar.set_postfix(G=f'{loss_G.item():.3f}', D=f'{loss_D.item():.3f}')

    # ── Update LR schedulers ──
    sched_G.step()
    sched_D.step()

    # ── Validation ──
    val_l1, val_psnr, val_ssim = validate()

    # ── Timing ──
    epoch_time   = time.time() - epoch_start
    elapsed      = time.time() - train_start
    remaining    = (elapsed / epoch) * (EPOCHS - epoch)
    def fmt(s): return f'{int(s//3600)}h{int((s%3600)//60)}m' if s>=3600 else f'{int(s//60)}m{int(s%60)}s'

    # ── Log ──
    n_batches = len(train_loader)
    g_avg = g_loss_sum / n_batches
    d_avg = d_loss_sum / n_batches
    is_best = val_psnr > best_psnr
    if is_best: best_psnr = val_psnr

    print(f'[{epoch:03d}/{EPOCHS}] '
          f'G={g_avg:.3f} D={d_avg:.3f} | '
          f'PSNR={val_psnr:.2f}dB SSIM={val_ssim:.4f} | '
          f'LR={sched_G.get_last_lr()[0]:.6f} | '
          f'{fmt(epoch_time)}/epoch | ETA={fmt(remaining)}'
          + (' ★BEST' if is_best else ''))

    # ── Save history ──
    history['epoch'].append(epoch)
    history['g_loss'].append(g_avg)
    history['d_loss'].append(d_avg)
    history['val_l1'].append(val_l1)
    history['psnr'].append(val_psnr)
    history['ssim'].append(val_ssim)

    # ── Save checkpoint ──
    if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
        save_checkpoint(epoch, g_avg, d_avg, val_psnr, val_ssim, is_best)
        save_samples(epoch)

total_time = time.time() - train_start
print('=' * 65)
print(f'Training complete in {fmt(total_time)}')
print(f'Best PSNR : {best_psnr:.2f} dB')
print(f'Best model: {BEST_MODEL}')

Starting training from epoch 1 to 50
Checkpoints every 5 epochs → /content/drive/MyDrive/Samudrika-v1/funiegan_training/checkpoints
Best model     → /content/drive/MyDrive/Samudrika-v1/funiegan_training/funiegan_best.pth


/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `peak_signal_noise_ratio` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `peak_signal_noise_ratio` from `torchmetrics.image` instead.
  _future_warning(
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


[001/50] G=23.740 D=0.372 | PSNR=21.49dB SSIM=0.8040 | LR=0.000200 | 4m34s/epoch | ETA=3h44m ★BEST


[002/50] G=19.343 D=0.271 | PSNR=21.73dB SSIM=0.8422 | LR=0.000200 | 4m41s/epoch | ETA=3h42m ★BEST


Epoch 003/50:  42%|████▏     | 216/517 [01:53<02:38,  1.90it/s, D=0.243, G=19.059]

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — Plot training curves
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['epoch'], history['g_loss'], label='G loss', color='blue')
axes[0].plot(history['epoch'], history['d_loss'], label='D loss', color='red')
axes[0].set_title('GAN Losses')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['epoch'], history['psnr'], color='green', marker='o', markersize=3)
axes[1].axhline(y=22, color='gray', linestyle='--', label='Target 22dB')
axes[1].set_title('Validation PSNR (target >22 dB)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history['epoch'], history['ssim'], color='purple', marker='o', markersize=3)
axes[2].axhline(y=0.80, color='gray', linestyle='--', label='Target 0.80')
axes[2].set_title('Validation SSIM (target >0.80)')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved → {OUTPUT_DIR}/training_curves.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — Final test set evaluation
# ═══════════════════════════════════════════════════════════
from torchmetrics.functional import (
    peak_signal_noise_ratio as psnr_fn,
    structural_similarity_index_measure as ssim_fn
)

# Load best model
best_ckpt = torch.load(BEST_MODEL, map_location=DEVICE)
generator.load_state_dict(best_ckpt['g_state'])
generator.eval()

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)

total_psnr, total_ssim, total_l1 = 0.0, 0.0, 0.0

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test evaluation'):
        real_A = batch['A'].to(DEVICE)
        real_B = batch['B'].to(DEVICE)
        fake_B = generator(real_A)
        fake_01 = (fake_B + 1) / 2
        real_01 = (real_B + 1) / 2
        total_psnr += psnr_fn(fake_01, real_01, data_range=1.0).item()
        total_ssim += ssim_fn(fake_01, real_01, data_range=1.0).item()
        total_l1   += criterion_l1(fake_B, real_B).item()

n = len(test_loader)
print('\n' + '='*50)
print('TEST SET RESULTS (best model)')
print('='*50)
print(f'PSNR : {total_psnr/n:.4f} dB  (target >22 dB)')
print(f'SSIM : {total_ssim/n:.4f}     (target >0.80)')
print(f'L1   : {total_l1/n:.4f)')
print('='*50)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — Export enhanced dataset for YOLOv11
# ═══════════════════════════════════════════════════════════
# This runs inference on ALL images (train+val+test) and saves
# enhanced versions → ready to be used as YOLO training images
from tqdm import tqdm
from torchvision import transforms
from PIL import Image
import torch

ENHANCED_DIR = OUTPUT_DIR / 'enhanced_dataset'
ENHANCED_DIR.mkdir(exist_ok=True)

denorm = transforms.Compose([
    transforms.Normalize((-1,-1,-1), (2,2,2)),  # [-1,1] → [0,1]
    transforms.ToPILImage()
])

transform_in = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

generator.eval()
all_images = sorted(list(dist_dir.glob('*')))
print(f'Enhancing {len(all_images)} images → {ENHANCED_DIR}')

with torch.no_grad():
    for img_path in tqdm(all_images):
        img = Image.open(img_path).convert('RGB')
        orig_size = img.size  # (W, H)
        inp = transform_in(img).unsqueeze(0).to(DEVICE)
        out = generator(inp).squeeze(0).cpu()
        enhanced = denorm(out)
        enhanced = enhanced.resize(orig_size, Image.LANCZOS)  # restore original size
        enhanced.save(ENHANCED_DIR / img_path.name)

enhanced_count = len(list(ENHANCED_DIR.glob('*')))
print(f'Done — {enhanced_count} enhanced images saved to {ENHANCED_DIR}')
print('These are ready to be used as YOLO training images in Phase 4.')